# Berkeley Parcels × Active Housing Permits

**Goal:** Identify which of Berkeley's 29,024 parcels currently have active housing-related permits.

Wed mar 18

**Data sources:**
- Parcels: `bhxd-e6up` (City of Berkeley Open Data)
- Zoning Permits: via Socrata API
- Local database: `/Users/johngage/berkeley-data/berkeley.db`

**Strategy:**
1. Load existing parcel data from local SQLite (fast) or re-fetch from API
2. Fetch all active/open housing-related permits from Berkeley Open Data
3. Join on APN to identify which parcels have active permits
4. Summarize and map the results

In [3]:
# CELL 1: Setup and imports
import pandas as pd
import requests
import sqlite3
import json
import os
from datetime import datetime

print(f"🕐 Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Berkeley Open Data Socrata API
BASE_URL = "https://data.cityofberkeley.info/resource"

# Dataset IDs
DATASETS = {
    'parcels': 'bhxd-e6up',
    'business_licenses': 'rwnf-bu3w',
}

# App token (optional but recommended to avoid throttling)
APP_TOKEN = os.environ.get('SOCRATA_APP_TOKEN', '')

# Local database path
DB_PATH = '/Users/johngage/berkeley-data/databases/berkeley.db'

print("✅ Imports loaded")
print(f"📁 Local DB: {DB_PATH}")
print(f"   Exists: {os.path.exists(DB_PATH)}")

🕐 Run started: 2026-03-18 17:30:26
✅ Imports loaded
📁 Local DB: /Users/johngage/berkeley-data/databases/berkeley.db
   Exists: True


In [4]:
# CELL 2: Load parcels from local database
print("📦 LOADING PARCELS FROM LOCAL DATABASE\n")
print("="*70)

conn = sqlite3.connect(DB_PATH)

# Check what tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables in database: {tables['name'].tolist()}\n")

# Load parcels
if 'parcels' in tables['name'].values:
    df_parcels = pd.read_sql('SELECT * FROM parcels', conn)
    print(f"✅ Loaded {len(df_parcels):,} parcels from local DB")
    print(f"📋 Columns: {df_parcels.columns.tolist()}")
    print(f"\nSample:")
    display(df_parcels.head(3))
else:
    print("❌ No 'parcels' table found. Will fetch from API in next cell.")
    df_parcels = None

conn.close()

📦 LOADING PARCELS FROM LOCAL DATABASE

Tables in database: ['licenses_fts', 'licenses_fts_data', 'licenses_fts_idx', 'licenses_fts_docsize', 'licenses_fts_config', 'parcels', 'corridor_ownership', 'corridor_far', 'rent_control', 'corridor_boundaries', 'licenses']

✅ Loaded 29,024 parcels from local DB
📋 Columns: ['SitusStree', 'the_geom', 'DATE_UPDAT', 'APN', 'SitusStr_1', 'SitusUnit', 'SitusCity', 'SitusZip', 'UseCode', 'BuildingAr', 'LotSize', 'SitusAddre', 'Longitude', 'Latitude', 'PARCELID', 'EXT_MIN_X', 'EXT_MIN_Y', 'EXT_MAX_X', 'EXT_MAX_Y', 'corridor']

Sample:


,SitusStree,the_geom,DATE_UPDAT,APN,SitusStr_1,SitusUnit,SitusCity,SitusZip,UseCode,BuildingAr,LotSize,SitusAddre,Longitude,Latitude,PARCELID,EXT_MIN_X,EXT_MIN_Y,EXT_MAX_X,EXT_MAX_Y,corridor
0,3208,MULTIPOLYGON (((-122.26620506714936 37.8520101...,2004-05-10,16-1428-2-2,SHATTUCK AVE,,BERKELEY,94705,2400,"2,395","1,995",3208 SHATTUCK AVE BERKELEY 94705,-122.26636568,37.85201812,016 142800202,"564,525.0572","4,189,644.3367","564,556.3459","4,189,655.4693",None
1,6618,MULTIPOLYGON (((-122.26520034486127 37.8521538...,2007-03-02,16-1425-57,SHATTUCK AVE,,BERKELEY,94609,8100,"5,000","1,000",6618 SHATTUCK AVE BERKELEY 94609,-122.2653821,37.85214246,016 142505700,"564,612.595","4,189,659.3389","564,643.192","4,189,669.0992",None
2,2320,MULTIPOLYGON (((-122.26080495483338 37.8527246...,2004-05-10,16-1422-22,WOOLSEY ST,,BERKELEY,94705,9300,0,"1,174",2320 WOOLSEY ST BERKELEY 94705,-122.26088766,37.85275106,016 142202200,"565,014.7806","4,189,729.858","565,029.3755","4,189,740.162",None


In [ ]:
# CELL 3: (If needed) Fetch all 29,024 parcels from API
# Skip this cell if Cell 2 loaded parcels successfully

if df_parcels is None or len(df_parcels) == 0:
    print("🌐 FETCHING ALL PARCELS FROM BERKELEY OPEN DATA API\n")
    print("="*70)
    
    all_parcels = []
    offset = 0
    limit = 1000
    
    while True:
        url = f"{BASE_URL}/{DATASETS['parcels']}.json"
        params = {
            '$limit': limit,
            '$offset': offset,
        }
        if APP_TOKEN:
            params['$$app_token'] = APP_TOKEN
        
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code != 200:
            print(f"❌ Error at offset {offset}: {response.status_code}")
            break
        
        batch = response.json()
        if not batch:
            break
        
        all_parcels.extend(batch)
        offset += limit
        print(f"  Fetched {len(all_parcels):,} parcels...")
        
        if len(batch) < limit:
            break
    
    df_parcels = pd.DataFrame(all_parcels)
    print(f"\n✅ Total parcels fetched: {len(df_parcels):,}")
    print(f"📋 Columns: {df_parcels.columns.tolist()}")
    
    # Save to local DB for next time
    conn = sqlite3.connect(DB_PATH)
    df_parcels.to_sql('parcels', conn, if_exists='replace', index=False)
    conn.close()
    print(f"💾 Saved to {DB_PATH}")
else:
    print(f"✅ Already have {len(df_parcels):,} parcels from local DB. Skipping API fetch.")

In [5]:
# CELL 4: Identify the APN column and standardize
print("🔧 STANDARDIZING APN COLUMN\n")
print("="*70)

# Find the APN column (could be 'APN', 'apn', 'parcel_no', etc.)
apn_candidates = [c for c in df_parcels.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
print(f"Possible APN columns: {apn_candidates}")

# Also check for address columns
addr_candidates = [c for c in df_parcels.columns if 'addr' in c.lower() or 'situs' in c.lower() or 'street' in c.lower()]
print(f"Possible address columns: {addr_candidates}")

# Set the APN column name (adjust if needed)
APN_COL = apn_candidates[0] if apn_candidates else None
ADDR_COL = addr_candidates[0] if addr_candidates else None

if APN_COL:
    print(f"\nUsing APN column: '{APN_COL}'")
    print(f"Sample APNs: {df_parcels[APN_COL].head(5).tolist()}")
    print(f"Unique APNs: {df_parcels[APN_COL].nunique():,}")
    print(f"Null APNs: {df_parcels[APN_COL].isna().sum():,}")
    
    # Standardize: strip whitespace, uppercase
    df_parcels['apn_clean'] = df_parcels[APN_COL].astype(str).str.strip().str.upper()
else:
    print("⚠️ No APN column found! Check column names above.")

if ADDR_COL:
    print(f"\nUsing Address column: '{ADDR_COL}'")
    print(f"Sample: {df_parcels[ADDR_COL].head(3).tolist()}")

🔧 STANDARDIZING APN COLUMN

Possible APN columns: ['APN', 'PARCELID']
Possible address columns: ['SitusStree', 'SitusStr_1', 'SitusUnit', 'SitusCity', 'SitusZip', 'SitusAddre']

Using APN column: 'APN'
Sample APNs: ['16-1428-2-2', '16-1425-57', '16-1422-22', '16-1422-20', '16-1422-24']
Unique APNs: 29,003
Null APNs: 0

Using Address column: 'SitusStree'
Sample: ['3208', '6618', '2320']


In [8]:
# CELL 5: Fetch active housing permits from Berkeley Open Data
# CELL 5: Query Berkeley's ArcGIS Planning/Accela MapServer for permit data
print("🏗️ QUERYING BERKELEY ARCGIS PLANNING SERVER\n")
print("="*70)

# Berkeley's Accela/Planning MapServer
ARCGIS_BASE = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Accela/MapServer"

# Key layers to explore:
#  0 = Building Inspection Areas
#  1 = Addresses
#  4 = Parcels 
# 24 = Zoning Districts
# 29 = RDA_FLAGS_ADDRESSES (permit flags by address)
# 30 = RDA_FLAGS_PARCEL (permit flags by parcel)

layers_to_check = {
    4: "Parcels",
    29: "RDA_FLAGS_ADDRESSES",
    30: "RDA_FLAGS_PARCEL",
    24: "Zoning Districts",
    1: "Addresses",
}

for layer_id, layer_name in layers_to_check.items():
    print(f"\n{'─'*60}")
    print(f"📋 Layer {layer_id}: {layer_name}")
    print(f"{'─'*60}")
    
    try:
        # First, get layer metadata to see fields
        meta_url = f"{ARCGIS_BASE}/{layer_id}?f=json"
        meta_resp = requests.get(meta_url, timeout=15)
        
        if meta_resp.status_code == 200:
            meta = meta_resp.json()
            fields = meta.get('fields', [])
            
            print(f"   Fields ({len(fields)}):")
            for f in fields:
                print(f"     • {f['name']} ({f['type']})")
            
            # Get a small sample of actual data
            query_url = f"{ARCGIS_BASE}/{layer_id}/query"
            params = {
                'where': '1=1',
                'outFields': '*',
                'returnGeometry': 'false',
                'resultRecordCount': 3,
                'f': 'json'
            }
            
            sample_resp = requests.get(query_url, params=params, timeout=15)
            
            if sample_resp.status_code == 200:
                sample_data = sample_resp.json()
                features = sample_data.get('features', [])
                
                if features:
                    print(f"\n   Sample records ({len(features)}):")
                    for feat in features:
                        attrs = feat.get('attributes', {})
                        for k, v in attrs.items():
                            print(f"     {k}: {v}")
                        print()
                
                # Get total count
                count_params = {
                    'where': '1=1',
                    'returnCountOnly': 'true',
                    'f': 'json'
                }
                count_resp = requests.get(query_url, params=count_params, timeout=15)
                if count_resp.status_code == 200:
                    total = count_resp.json().get('count', '?')
                    print(f"   📊 Total records: {total:,}" if isinstance(total, int) else f"   📊 Total records: {total}")
            else:
                print(f"   ⚠️ Query failed: HTTP {sample_resp.status_code}")
        else:
            print(f"   ⚠️ Metadata failed: HTTP {meta_resp.status_code}")
    
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n{'='*70}")
print("\n✅ Review the fields above to see which layers contain permit data.")
print("   Layer 29 (RDA_FLAGS_ADDRESSES) and 30 (RDA_FLAGS_PARCEL)")
print("   are the most likely to contain active permit information.")

🏗️ QUERYING BERKELEY ARCGIS PLANNING SERVER


────────────────────────────────────────────────────────────
📋 Layer 4: Parcels
────────────────────────────────────────────────────────────
   Fields (60):
     • OBJECTID (esriFieldTypeOID)
     • APN (esriFieldTypeString)
     • APN_SORT (esriFieldTypeString)
     • DATE_UPDAT (esriFieldTypeString)
     • SitusStree (esriFieldTypeString)
     • SitusStr_1 (esriFieldTypeString)
     • SitusUnit (esriFieldTypeString)
     • SitusCity (esriFieldTypeString)
     • SitusZip (esriFieldTypeString)
     • Land (esriFieldTypeDouble)
     • Imps (esriFieldTypeDouble)
     • HOEX (esriFieldTypeDouble)
     • OTEX (esriFieldTypeDouble)
     • TotalNetVa (esriFieldTypeDouble)
     • LatestDocu (esriFieldTypeString)
     • OwnersName (esriFieldTypeString)
     • MailingAdd (esriFieldTypeString)
     • MailingA_1 (esriFieldTypeString)
     • MailingA_2 (esriFieldTypeString)
     • MailingA_3 (esriFieldTypeString)
     • MailingA_4 (esriFieldTypeString)

In [9]:
# CELL A: Fetch ALL addresses from ArcGIS Layer 1
print("🌐 FETCHING ALL ADDRESSES FROM ARCGIS\n")
print("="*70)

ARCGIS_BASE = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Accela/MapServer"

all_addr = []
offset = 0

while True:
    params = {
        'where': '1=1',
        'outFields': '*',
        'returnGeometry': 'false',
        'resultOffset': offset,
        'resultRecordCount': 2000,
        'f': 'json'
    }
    resp = requests.get(f"{ARCGIS_BASE}/1/query", params=params, timeout=30)
    features = resp.json().get('features', [])
    if not features:
        break
    all_addr.extend([f['attributes'] for f in features])
    offset += len(features)
    print(f"  Fetched {len(all_addr):,} addresses...")
    import time; time.sleep(0.3)

df_addresses = pd.DataFrame(all_addr)
print(f"\n✅ Total addresses: {len(df_addresses):,}")
print(f"Columns: {df_addresses.columns.tolist()}")

🌐 FETCHING ALL ADDRESSES FROM ARCGIS

  Fetched 2,000 addresses...
  Fetched 4,000 addresses...
  Fetched 6,000 addresses...
  Fetched 8,000 addresses...
  Fetched 10,000 addresses...
  Fetched 12,000 addresses...
  Fetched 14,000 addresses...
  Fetched 16,000 addresses...
  Fetched 18,000 addresses...
  Fetched 20,000 addresses...
  Fetched 22,000 addresses...
  Fetched 24,000 addresses...
  Fetched 26,000 addresses...
  Fetched 28,000 addresses...
  Fetched 30,000 addresses...
  Fetched 32,000 addresses...
  Fetched 34,000 addresses...
  Fetched 36,000 addresses...
  Fetched 38,000 addresses...
  Fetched 40,000 addresses...
  Fetched 42,000 addresses...
  Fetched 44,000 addresses...
  Fetched 46,000 addresses...
  Fetched 48,000 addresses...
  Fetched 50,000 addresses...
  Fetched 52,000 addresses...
  Fetched 54,000 addresses...
  Fetched 56,000 addresses...
  Fetched 58,000 addresses...
  Fetched 60,000 addresses...
  Fetched 62,000 addresses...
  Fetched 64,000 addresses...
  Fetc

In [10]:
# CELL B: Normalize APNs and join parcels ↔ addresses
print("🔗 NORMALIZING APNs AND JOINING\n")
print("="*70)

import re

def normalize_apn(apn):
    """Strip dashes, spaces, leading zeros → pure digits for comparison"""
    if pd.isna(apn):
        return ''
    s = re.sub(r'[\s\-\.]+', '', str(apn).strip())
    return s.lstrip('0') or '0'

# Normalize parcels
df_parcels['apn_norm'] = df_parcels['APN'].apply(normalize_apn)

# Normalize addresses
df_addresses['apn_norm'] = df_addresses['APN'].apply(normalize_apn)

print(f"Parcels unique normalized APNs: {df_parcels['apn_norm'].nunique():,}")
print(f"Addresses unique normalized APNs: {df_addresses['apn_norm'].nunique():,}")

# JOIN by normalized APN
df_joined = df_parcels.merge(
    df_addresses,
    on='apn_norm',
    how='left',
    suffixes=('_parcel', '_addr'),
    indicator=True
)

matched = (df_joined['_merge'] == 'both').sum()
unmatched = (df_joined['_merge'] == 'left_only').sum()

print(f"\n✅ Parcels with matching address: {matched:,}")
print(f"❌ Parcels with NO match:         {unmatched:,}")
print(f"   Match rate: {matched/(matched+unmatched)*100:.1f}%")

# Save to SQLite
conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')
df_parcels.to_sql('parcels_arcgis', conn, if_exists='replace', index=False)
df_addresses.to_sql('addresses_arcgis', conn, if_exists='replace', index=False)
df_joined.to_sql('parcels_addresses_joined', conn, if_exists='replace', index=False)

# Create indexes
conn.execute('CREATE INDEX IF NOT EXISTS idx_p_apn ON parcels_arcgis(apn_norm)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_a_apn ON addresses_arcgis(apn_norm)')
conn.commit()
conn.close()

print(f"\n💾 Saved 3 tables to berkeley.db:")
print(f"   • parcels_arcgis: {len(df_parcels):,} rows")
print(f"   • addresses_arcgis: {len(df_addresses):,} rows")
print(f"   • parcels_addresses_joined: {len(df_joined):,} rows")

🔗 NORMALIZING APNs AND JOINING

Parcels unique normalized APNs: 28,694
Addresses unique normalized APNs: 29,170

✅ Parcels with matching address: 0
❌ Parcels with NO match:         29,024
   Match rate: 0.0%


OperationalError: duplicate column name: ParcelID

In [12]:
# CELL: Fix and save joined table — drop conflicting columns before saving
print("🔧 Saving joined table (handling duplicate columns)...\n")

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# The parcels and addresses tables saved fine — they're already in the DB.
# For the joined table, we'll use SQL instead of pandas to avoid the column conflict.

# Create the join as a SQL VIEW instead of a table — cleaner and no duplication issues
conn.execute("DROP VIEW IF EXISTS parcels_addresses_joined")
conn.execute("""
    CREATE VIEW parcels_addresses_joined AS
    SELECT 
        p.*,
        a.LocationID,
        a.FullAddress AS addr_FullAddress,
        a.StreetNumber,
        a.StreetName,
        a.StreetSuffix,
        a.Unit AS addr_Unit,
        a.ZipCode AS addr_ZipCode,
        a.OwnerName AS addr_OwnerName,
        a.OwnerAddress1,
        a.OwnerAddress2,
        a.ownercityst,
        a.OwnerZip AS addr_OwnerZip,
        a.AddressType,
        a.Status AS addr_Status,
        a.BldgSqft AS addr_BldgSqft,
        a.LotSqft AS addr_LotSqft,
        a.InDbDate
    FROM parcels_arcgis p
    LEFT JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
""")
conn.commit()

# Check match rate
stats = pd.read_sql("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN a.apn_norm IS NOT NULL THEN 1 ELSE 0 END) as matched,
        SUM(CASE WHEN a.apn_norm IS NULL THEN 1 ELSE 0 END) as unmatched
    FROM parcels_arcgis p
    LEFT JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
""", conn)

total = stats.iloc[0]['total_rows']
matched = stats.iloc[0]['matched']
unmatched = stats.iloc[0]['unmatched']

print(f"✅ Created SQL VIEW 'parcels_addresses_joined'")
print(f"\n   Total rows:     {total:,}")
print(f"   Matched:        {matched:,}")
print(f"   Unmatched:      {unmatched:,}")
print(f"   Match rate:     {matched/total*100:.1f}%")

# Also check for owner name mismatches between the two sources
mismatches = pd.read_sql("""
    SELECT COUNT(*) as n
    FROM parcels_arcgis p
    JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
    WHERE p.OwnersName IS NOT NULL 
      AND a.OwnerName IS NOT NULL
      AND UPPER(TRIM(p.OwnersName)) != UPPER(TRIM(a.OwnerName))
""", conn)
print(f"\n   Owner name mismatches: {mismatches.iloc[0]['n']:,}")

# Create indexes
conn.execute('CREATE INDEX IF NOT EXISTS idx_p_apn ON parcels_arcgis(apn_norm)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_a_apn ON addresses_arcgis(apn_norm)')
conn.commit()

# Show all t

🔧 Saving joined table (handling duplicate columns)...

✅ Created SQL VIEW 'parcels_addresses_joined'

   Total rows:     29,024
   Matched:        0
   Unmatched:      29,024
   Match rate:     0.0%


DatabaseError: Execution failed on sql '
    SELECT COUNT(*) as n
    FROM parcels_arcgis p
    JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
    WHERE p.OwnersName IS NOT NULL 
      AND a.OwnerName IS NOT NULL
      AND UPPER(TRIM(p.OwnersName)) != UPPER(TRIM(a.OwnerName))
': no such column: p.OwnersName

In [13]:
# CELL: Debug — check actual APNs and column names
print("🔍 DEBUGGING APN MATCH FAILURE\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# 1. Check actual column names in both tables
print("PARCELS columns:")
p_cols = pd.read_sql("PRAGMA table_info(parcels_arcgis)", conn)
for _, row in p_cols.iterrows():
    print(f"   {row['name']}")

print("\nADDRESSES columns:")
a_cols = pd.read_sql("PRAGMA table_info(addresses_arcgis)", conn)
for _, row in a_cols.iterrows():
    print(f"   {row['name']}")

# 2. Show sample APNs from both tables BEFORE and AFTER normalization
print("\n" + "="*70)
print("\nSample PARCEL APNs (raw → normalized):")
samples_p = pd.read_sql("SELECT APN, apn_norm FROM parcels_arcgis LIMIT 10", conn)
for _, row in samples_p.iterrows():
    print(f"   '{row['APN']}' → '{row['apn_norm']}'")

print("\nSample ADDRESS APNs (raw → normalized):")
samples_a = pd.read_sql("SELECT APN, apn_norm FROM addresses_arcgis LIMIT 10", conn)
for _, row in samples_a.iterrows():
    print(f"   '{row['APN']}' → '{row['apn_norm']}'")

# 3. Try to find ANY overlap
print("\n" + "="*70)
overlap = pd.read_sql("""
    SELECT COUNT(*) as n FROM (
        SELECT DISTINCT apn_norm FROM parcels_arcgis
        INTERSECT
        SELECT DISTINCT apn_norm FROM addresses_arcgis
    )
""", conn)
print(f"\nOverlapping apn_norm values: {overlap.iloc[0]['n']:,}")

# 4. If no overlap, try raw APN match
overlap_raw = pd.read_sql("""
    SELECT COUNT(*) as n FROM (
        SELECT DISTINCT APN FROM parcels_arcgis
        INTERSECT
        SELECT DISTINCT APN FROM addresses_arcgis
    )
""", conn)
print(f"Overlapping raw APN values:  {overlap_raw.iloc[0]['n']:,}")

conn.close()

🔍 DEBUGGING APN MATCH FAILURE

PARCELS columns:
   SitusStree
   the_geom
   DATE_UPDAT
   APN
   SitusStr_1
   SitusUnit
   SitusCity
   SitusZip
   UseCode
   BuildingAr
   LotSize
   SitusAddre
   Longitude
   Latitude
   PARCELID
   EXT_MIN_X
   EXT_MIN_Y
   EXT_MAX_X
   EXT_MAX_Y
   corridor
   apn_clean
   apn_norm

ADDRESSES columns:
   OBJECTID
   LocationID
   LocationName
   ParcelID
   APN
   Status
   AddressType
   StreetNumber
   PreQualifier
   Direction
   StreetName
   Unit
   StreetSuffix
   ZipCode
   FullAddress
   UseCode
   LotSqft
   BldgSqft
   OwnerName
   OwnerAddress1
   OwnerAddress2
   ownercityst
   OwnerZip
   U_X
   U_Y
   InDbDate
   apn_norm


Sample PARCEL APNs (raw → normalized):
   '16-1428-2-2' → '16142822'
   '16-1425-57' → '16142557'
   '16-1422-22' → '16142222'
   '16-1422-20' → '16142220'
   '16-1422-24' → '16142224'
   '16-1422-26' → '16142226'
   '16-1422-28' → '16142228'
   '16-1422-30' → '16142230'
   '16-1422-32' → '16142232'
   '16-1414-4

In [14]:
# CELL: Fix APN normalization — align the two different formats
print("🔧 FIXING APN FORMAT ALIGNMENT\n")
print("="*70)

import re

def normalize_parcel_apn(apn):
    """Parcel format: 16-1422-22 or 16-1410-1-1"""
    if pd.isna(apn):
        return ''
    s = str(apn).strip()
    # Split on dashes
    parts = s.split('-')
    if len(parts) >= 3:
        # Pad: book(3) map(4) parcel(3) sub(2)
        book = parts[0].zfill(3)
        mapnum = parts[1].zfill(4)
        parcel = parts[2].zfill(3)
        sub = parts[3].zfill(2) if len(parts) > 3 else '00'
        return f"{book}{mapnum}{parcel}{sub}"
    return re.sub(r'[\s\-]+', '', s)

def normalize_address_apn(apn):
    """Address format: 016 142202200 (already book+map+parcel+sub as one block)"""
    if pd.isna(apn):
        return ''
    s = re.sub(r'[\s\-]+', '', str(apn).strip())
    # Should already be 12 digits: 3(book) + 4(map) + 3(parcel) + 2(sub)
    return s.lstrip('0').zfill(12) if s else ''

# Apply to parcels
df_parcels['apn_norm'] = df_parcels['APN'].apply(normalize_parcel_apn)

# Apply to addresses  
df_addresses['apn_norm'] = df_addresses['APN'].apply(normalize_address_apn)

# Show samples to verify alignment
print("PARCEL APNs (raw → normalized):")
for _, row in df_parcels[['APN', 'apn_norm']].head(10).iterrows():
    print(f"   '{row['APN']}' → '{row['apn_norm']}'")

print("\nADDRESS APNs (raw → normalized):")
for _, row in df_addresses[['APN', 'apn_norm']].head(10).iterrows():
    print(f"   '{row['APN']}' → '{row['apn_norm']}'")

# Check overlap now
parcel_apns = set(df_parcels['apn_norm'].unique())
address_apns = set(df_addresses['apn_norm'].unique())
overlap = parcel_apns & address_apns

print(f"\n{'='*70}")
print(f"\n   Unique parcel APNs:  {len(parcel_apns):,}")
print(f"   Unique address APNs: {len(address_apns):,}")
print(f"   OVERLAP:             {len(overlap):,}")

if overlap:
    print(f"   ✅ Match rate: {len(overlap)/len(parcel_apns)*100:.1f}% of parcels")
    print(f"\n   Sample matches:")
    for apn in list(overlap)[:5]:
        p_raw = df_parcels[df_parcels['apn_norm']==apn]['APN'].iloc[0]
        a_raw = df_addresses[df_addresses['apn_norm']==apn]['APN'].iloc[0]
        print(f"      Parcel '{p_raw}' ↔ Address '{a_raw}' → '{apn}'")
else:
    print("   ❌ Still no matches. Let's compare digit by digit...")
    # Show side by side for manual comparison
    p_sample = df_parcels[['APN','apn_norm']].head(5)
    a_sample = df_addresses[['APN','apn_norm']].head(5)
    print(f"\n   Parcel norms:  {p_sample['apn_norm'].tolist()}")
    print(f"   Address norms: {a_sample['apn_norm'].tolist()}")
    print(f"\n   Parcel norm lengths:  {[len(x) for x in p_sample['apn_norm'].tolist()]}")
    print(f"   Address norm lengths: {[len(x) for x in a_sample['apn_norm'].tolist()]}")

# Update SQLite
conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')
df_parcels.to_sql('parcels_arcgis', conn, if_exists='replace', index=False)
df_addresses.to_sql('addresses_arcgis', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX IF NOT EXISTS idx_p_apn ON parcels_arcgis(apn_norm)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_a_apn ON addresses_arcgis(apn_norm)')
conn.commit()
conn.close()
print("\n💾 Updated both tables in SQLite")

🔧 FIXING APN FORMAT ALIGNMENT

PARCEL APNs (raw → normalized):
   '16-1428-2-2' → '016142800202'
   '16-1425-57' → '016142505700'
   '16-1422-22' → '016142202200'
   '16-1422-20' → '016142202000'
   '16-1422-24' → '016142202400'
   '16-1422-26' → '016142202600'
   '16-1422-28' → '016142202800'
   '16-1422-30' → '016142203000'
   '16-1422-32' → '016142203200'
   '16-1414-43' → '016141404300'

ADDRESS APNs (raw → normalized):
   '016 142505700' → '016142505700'
   '016 142202200' → '016142202200'
   '016 142202000' → '016142202000'
   '016 142202000' → '016142202000'
   '016 142202000' → '016142202000'
   '016 142202000' → '016142202000'
   '016 142202000' → '016142202000'
   '016 141003200' → '016141003200'
   '016 141001501' → '016141001501'
   '016 141001801' → '016141001801'


   Unique parcel APNs:  29,003
   Unique address APNs: 29,170
   OVERLAP:             28,978
   ✅ Match rate: 99.9% of parcels

   Sample matches:
      Parcel '60-2392-43' ↔ Address '060 239204300' → '06023920

In [16]:
# CELL: Rebuild the joined view and get summary stats
print("🔗 REBUILDING JOIN VIEW WITH CORRECT COLUMNS\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Drop old view and recreate with actual column names
conn.execute("DROP VIEW IF EXISTS parcels_addresses_joined")
conn.execute("""
    CREATE VIEW parcels_addresses_joined AS
    SELECT 
        p.APN as parcel_APN,
        p.apn_norm,
        p.SitusAddre,
        p.SitusStree,
        p.SitusStr_1,
        p.SitusUnit,
        p.SitusCity,
        p.SitusZip,
        p.UseCode as parcel_UseCode,
        p.LotSize as parcel_LotSize,
        p.Latitude,
        p.Longitude,
        p.PARCELID,
        p.corridor,
        a.FullAddress,
        a.StreetNumber,
        a.StreetName,
        a.StreetSuffix,
        a.Unit as addr_Unit,
        a.ZipCode as addr_ZipCode,
        a.OwnerName,
        a.OwnerAddress1,
        a.OwnerAddress2,
        a.ownercityst,
        a.OwnerZip,
        a.UseCode as addr_UseCode,
        a.LotSqft as addr_LotSqft,
        a.BldgSqft as addr_BldgSqft,
        a.AddressType,
        a.Status as addr_Status
    FROM parcels_arcgis p
    LEFT JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
""")
conn.commit()

# Summary
print("📊 JOINED DATABASE SUMMARY:\n")

total = pd.read_sql("SELECT COUNT(*) as n FROM parcels_addresses_joined", conn).iloc[0]['n']
matched = pd.read_sql("SELECT COUNT(*) as n FROM parcels_addresses_joined WHERE FullAddress IS NOT NULL", conn).iloc[0]['n']
with_owner = pd.read_sql("SELECT COUNT(*) as n FROM parcels_addresses_joined WHERE OwnerName IS NOT NULL AND OwnerName != ''", conn).iloc[0]['n']

print(f"   Total rows:          {total:,}")
print(f"   With address match:  {matched:,} ({matched/total*100:.1f}%)")
print(f"   With owner name:     {with_owner:,} ({with_owner/total*100:.1f}%)")

# Use code breakdown
print(f"\n📋 USE CODE DISTRIBUTION (top 15):")
use_codes = pd.read_sql("""
    SELECT parcel_UseCode, COUNT(*) as count
    FROM parcels_addresses_joined
    GROUP BY parcel_UseCode
    ORDER BY count DESC
    LIMIT 15
""", conn)
for _, row in use_codes.iterrows():
    print(f"   {row['parcel_UseCode'] or 'NULL':>6}: {row['count']:,} parcels")

# Unmatched parcels
print(f"\n❌ UNMATCHED PARCELS (no address found):")
unmatched = pd.read_sql("""
    SELECT parcel_APN, SitusAddre
    FROM parcels_addresses_joined
    WHERE FullAddress IS NULL
    LIMIT 10
""", conn)
for _, row in unmatched.iterrows():
    print(f"   APN: {row['parcel_APN']} → {row['SitusAddre']}")

conn.close()
print(f"\n✅ View 'parcels_addresses_joined' ready to query!")
print(f"   Database: /Users/johngage/berkeley-data/databases/berkeley.db")

🔗 REBUILDING JOIN VIEW WITH CORRECT COLUMNS

📊 JOINED DATABASE SUMMARY:

   Total rows:          65,297
   With address match:  65,271 (100.0%)
   With owner name:     64,983 (99.5%)

📋 USE CODE DISTRIBUTION (top 15):
     1100: 17,257 parcels
     7700: 17,142 parcels
     3200: 2,806 parcels
     2400: 2,449 parcels
     2200: 2,216 parcels
     7300: 2,137 parcels
     2500: 1,819 parcels
     0300: 1,596 parcels
     2100: 1,447 parcels
     2300: 1,390 parcels
     3100: 1,140 parcels
     2600: 1,119 parcels
     9400: 1,068 parcels
     7200: 955 parcels
     7500: 907 parcels

❌ UNMATCHED PARCELS (no address found):
   APN: 16-1410-2-1 → 2634 WOOLSEY ST BERKELEY 94705
   APN: 55-1839- → 
   APN: 55-1839- → 
   APN: 52-1516-24 → 1310 HASKELL ST BERKELEY 94702
   APN: 54-1742-31-1 → 2747 SAN PABLO AVE BERKELEY 94702
   APN: 55-1876-21 → 2542 DURANT AVE BERKELEY 94704
   APN: 55-1876-20 → 2538 DURANT AVE BERKELEY 94704
   APN: 55-1864-10 → 2811 CHANNING WAY BERKELEY 94704
   APN: 

In [17]:
# CELL: Fetch Zoning Districts (Layer 24) from ArcGIS
print("🏗️ FETCHING ZONING DISTRICTS FROM ARCGIS\n")
print("="*70)

ARCGIS_BASE = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Accela/MapServer"

# Get metadata
meta = requests.get(f"{ARCGIS_BASE}/24?f=json", timeout=15).json()
print("Zoning layer fields:")
for f in meta.get('fields', []):
    print(f"   {f['name']} ({f['type']})")

# Get count
query_url = f"{ARCGIS_BASE}/24/query"
count = requests.get(query_url, params={'where':'1=1','returnCountOnly':'true','f':'json'}, timeout=15).json().get('count',0)
print(f"\nTotal zoning polygons: {count}")

# Fetch all
all_zones = []
offset = 0
while offset < count:
    params = {
        'where': '1=1',
        'outFields': '*',
        'returnGeometry': 'false',
        'resultOffset': offset,
        'resultRecordCount': 2000,
        'f': 'json'
    }
    resp = requests.get(query_url, params=params, timeout=30)
    features = resp.json().get('features', [])
    if not features:
        break
    all_zones.extend([f['attributes'] for f in features])
    offset += len(features)
    print(f"  Fetched {len(all_zones)} zones...")
    time.sleep(0.3)

df_zones = pd.DataFrame(all_zones)
print(f"\n✅ Zoning districts: {len(df_zones)} records")
print(f"Columns: {df_zones.columns.tolist()}")

# Show distribution
zone_col = [c for c in df_zones.columns if 'zone' in c.lower() or 'zoning' in c.lower() or 'district' in c.lower()]
print(f"\nPossible zone columns: {zone_col}")
if zone_col:
    print(f"\n📋 Unique zones:")
    for z, cnt in df_zones[zone_col[0]].value_counts().items():
        print(f"   {z}: {cnt}")

# Save
conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')
df_zones.to_sql('zoning_districts', conn, if_exists='replace', index=False)
conn.close()
print(f"\n💾 Saved to database")
display(df_zones.head(3))

🏗️ FETCHING ZONING DISTRICTS FROM ARCGIS

Zoning layer fields:
   OBJECTID (esriFieldTypeOID)
   ZONECLASS (esriFieldTypeString)
   ZONEDESC (esriFieldTypeString)
   BASEELEV (esriFieldTypeDouble)
   HEIGHT (esriFieldTypeDouble)
   LASTUPDATE (esriFieldTypeDate)
   LASTEDITOR (esriFieldTypeString)
   sde_prod.DBO.ZoningDistrict.AREA (esriFieldTypeDouble)
   PERIMETER (esriFieldTypeDouble)
   BLOCKS_ID (esriFieldTypeDouble)
   NEW_BLOCK (esriFieldTypeString)
   ZONE (esriFieldTypeString)
   GENPLAN (esriFieldTypeString)
   Shape (esriFieldTypeGeometry)
   Shape.STArea() (esriFieldTypeDouble)
   Shape.STLength() (esriFieldTypeDouble)

Total zoning polygons: 42
  Fetched 42 zones...

✅ Zoning districts: 42 records
Columns: ['OBJECTID', 'ZONECLASS', 'ZONEDESC', 'BASEELEV', 'HEIGHT', 'LASTUPDATE', 'LASTEDITOR', 'sde_prod.DBO.ZoningDistrict.AREA', 'PERIMETER', 'BLOCKS_ID', 'NEW_BLOCK', 'ZONE', 'GENPLAN', 'Shape.STArea()', 'Shape.STLength()']

Possible zone columns: ['ZONECLASS', 'ZONEDESC', 

,OBJECTID,ZONECLASS,ZONEDESC,BASEELEV,HEIGHT,LASTUPDATE,LASTEDITOR,sde_prod.DBO.ZoningDistrict.AREA,PERIMETER,BLOCKS_ID,NEW_BLOCK,ZONE,GENPLAN,Shape.STArea(),Shape.STLength()
0,1,C-AC,Adeline Corridor Commercial,0.0,0.0,NaN,,1.423834e+07,298411.951,1020.0,055 1826,R,NC,165675.314209,10794.715548
1,2,C-C,Corridor Commercial,0.0,0.0,NaN,,1.330578e+07,293079.706,618.0,058 2177,C,I,97448.321777,7274.751913
2,3,C-DMU Buffer,C-DMU Buffer,0.0,0.0,NaN,,6.440704e+06,93393.032,0.0,057 2059,R,MDR,94258.730957,5190.702972


In [18]:
# CELL: Build development potential for every parcel using zoning data
print("🏗️ BUILDING DEVELOPMENT POTENTIAL COLUMN\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Check what we got in zoning
df_zones = pd.read_sql("SELECT * FROM zoning_districts LIMIT 5", conn)
print("Zoning columns:", df_zones.columns.tolist())
print("\nSample:")
display(df_zones.head(3))

# Get unique zone classes
zone_dist = pd.read_sql("""
    SELECT ZONECLASS, ZONEDESC, COUNT(*) as polygon_count 
    FROM zoning_districts 
    GROUP BY ZONECLASS, ZONEDESC 
    ORDER BY polygon_count DESC
""", conn)
print(f"\n📋 ZONING DISTRICTS ({len(zone_dist)} types):\n")
for _, row in zone_dist.iterrows():
    print(f"   {row['ZONECLASS']:>10}: {row['ZONEDESC'] or 'N/A'} ({row['polygon_count']} polygons)")

conn.close()

🏗️ BUILDING DEVELOPMENT POTENTIAL COLUMN

Zoning columns: ['OBJECTID', 'ZONECLASS', 'ZONEDESC', 'BASEELEV', 'HEIGHT', 'LASTUPDATE', 'LASTEDITOR', 'sde_prod.DBO.ZoningDistrict.AREA', 'PERIMETER', 'BLOCKS_ID', 'NEW_BLOCK', 'ZONE', 'GENPLAN', 'Shape.STArea()', 'Shape.STLength()']

Sample:


,OBJECTID,ZONECLASS,ZONEDESC,BASEELEV,HEIGHT,LASTUPDATE,LASTEDITOR,sde_prod.DBO.ZoningDistrict.AREA,PERIMETER,BLOCKS_ID,NEW_BLOCK,ZONE,GENPLAN,Shape.STArea(),Shape.STLength()
0,1,C-AC,Adeline Corridor Commercial,0.0,0.0,None,,1.423834e+07,298411.951,1020.0,055 1826,R,NC,165675.314209,10794.715548
1,2,C-C,Corridor Commercial,0.0,0.0,None,,1.330578e+07,293079.706,618.0,058 2177,C,I,97448.321777,7274.751913
2,3,C-DMU Buffer,C-DMU Buffer,0.0,0.0,None,,6.440704e+06,93393.032,0.0,057 2059,R,MDR,94258.730957,5190.702972



📋 ZONING DISTRICTS (41 types):

          R-2: Multi-Unit 2 (2 polygons)
         C-AC: Adeline Corridor Commercial (1 polygons)
          C-C: Corridor Commercial (1 polygons)
   C-DMU Buffer: C-DMU Buffer (1 polygons)
   C-DMU Core: C-DMU Core (1 polygons)
   C-DMU Corridor: C-DMU Corridor (1 polygons)
   C-DMU Outer Core: C-DMU Outer Core (1 polygons)
          C-E: Elmwood Commercial (1 polygons)
          C-N: Neighborhood Commercial (1 polygons)
       C-N(H): Neighborhood Commercial (Hillside Overlay) (1 polygons)
         C-NS: North Shattuck Commercial (1 polygons)
      C-NS(H): North Shattuck Commercial (Hillside Overlay) (1 polygons)
         C-SA: South Area Commercial (1 polygons)
         C-SO: Solano Avenue Commercial (1 polygons)
          C-T: Telegraph Avenue Commercial (1 polygons)
          C-U: University Avenue Commercial (1 polygons)
          C-W: West Berkeley Commercial (1 polygons)
         ES-R: Environmental Safety-Residential (1 polygons)
            M: 

In [19]:
# CELL: Create development potential for every parcel
print("🏗️ ASSIGNING DEVELOPMENT POTENTIAL TO EVERY PARCEL\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Development potential rules based on middle housing ordinance + density bonus
# Effective Nov 1, 2025 (Ordinance 7,978-N.S.)
# Density Bonus per BMC 23.330 / Gov Code 65915

zone_potential = {
    # RESIDENTIAL - Middle Housing applies (flatlands)
    'R-1':    {'base_units': 8, 'max_stories': 3, 'middle_housing': True,  'hills': False, 'density_bonus': True,  'desc': 'Multi-Unit 1 — up to 8 units by right + ADUs'},
    'R-2':    {'base_units': 8, 'max_stories': 3, 'middle_housing': True,  'hills': False, 'density_bonus': True,  'desc': 'Multi-Unit 2 — up to 8 units by right + ADUs'},
    'R-2A':   {'base_units': 8, 'max_stories': 3, 'middle_housing': True,  'hills': False, 'density_bonus': True,  'desc': 'Multi-Unit 2A — up to 8 units by right + ADUs'},
    'MUR':    {'base_units': 8, 'max_stories': 3, 'middle_housing': True,  'hills': False, 'density_bonus': True,  'desc': 'Mixed Use-Residential — multi-unit by right'},

    # RESIDENTIAL - Hillside Overlay (restricted)
    'R-1H':   {'base_units': 1, 'max_stories': 2, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'SFR Hillside — middle housing restricted, fire hazard'},
    'R-2H':   {'base_units': 2, 'max_stories': 2, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'Two-Family Hillside — limited density'},
    'R-2AH':  {'base_units': 2, 'max_stories': 2, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'Restricted Multi-Family Hillside'},
    'R-3H':   {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'Multiple-Family Hillside — density per lot size'},
    'R-4H':   {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'Multi-Family Hillside — density per lot size'},
    'R-5H':   {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'High Density Hillside'},
    'R-SH':   {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'High Density Subarea Hillside'},

    # RESIDENTIAL - Higher density (flatlands)
    'R-3':    {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Multiple-Family — density per lot size, no unit cap'},
    'R-4':    {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Multi-Family — higher density per lot size'},
    'R-5':    {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'High Density Residential — apartments, hotels'},
    'R-S':    {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'High Density Subarea (Southside)'},
    'R-SMU':  {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Residential Southside Mixed Use'},
    'R-BMU':  {'base_units': None, 'max_stories': 7, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'BART Mixed Use — Ashby/North Berkeley stations'},

    # COMMERCIAL — residential allowed above ground floor or mixed use
    'C-AC':   {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Adeline Corridor — mixed use with housing'},
    'C-C':    {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Corridor Commercial — housing above ground floor'},
    'C-DMU Buffer':    {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Downtown Mixed Use Buffer — housing allowed'},
    'C-DMU Core':      {'base_units': None, 'max_stories': 10, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Downtown Core — high-rise housing allowed'},
    'C-DMU Corridor':  {'base_units': None, 'max_stories': 7, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Downtown Corridor — mid-rise housing'},
    'C-DMU Outer Core': {'base_units': None, 'max_stories': 7, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Downtown Outer Core — mid-rise housing'},
    'C-E':    {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Elmwood Commercial — housing above retail'},
    'C-N':    {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Neighborhood Commercial — housing above retail'},
    'C-N(H)': {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'Neighborhood Commercial Hillside'},
    'C-NS':   {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'North Shattuck — housing above retail'},
    'C-NS(H)':{'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': True,  'density_bonus': True,  'desc': 'North Shattuck Hillside'},
    'C-SA':   {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'South Area Commercial — mixed use'},
    'C-SO':   {'base_units': None, 'max_stories': 3, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Solano Ave — housing above retail'},
    'C-T':    {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Telegraph Ave — mixed use with housing'},
    'C-U':    {'base_units': None, 'max_stories': 5, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'University Ave — mixed use with housing'},
    'C-W':    {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'West Berkeley Commercial — mixed use allowed'},

    # INDUSTRIAL / MANUFACTURING
    'M':      {'base_units': 0, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': False, 'desc': 'Manufacturing — no housing allowed'},
    'MM':     {'base_units': 0, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': False, 'desc': 'Mixed Manufacturing — limited live/work only'},
    'MRD':    {'base_units': 0, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': False, 'desc': 'Manufacturing Research — no housing'},
    'MULI':   {'base_units': None, 'max_stories': 4, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Mixed Use-Light Industrial — some housing allowed'},

    # OTHER
    'ES-R':   {'base_units': 1, 'max_stories': 2, 'middle_housing': False, 'hills': True,  'density_bonus': False, 'desc': 'Environmental Safety-Residential — very restricted'},
    'SP':     {'base_units': None, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': True,  'desc': 'Specific Plan — varies by plan'},
    'U':      {'base_units': 0, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': False, 'desc': 'Unclassified'},
    'X':      {'base_units': 0, 'max_stories': None, 'middle_housing': False, 'hills': False, 'density_bonus': False, 'desc': 'Green Space — no development'},
}

# Save as a table
rows = []
for zone, info in zone_potential.items():
    rows.append({
        'zone_class': zone,
        'base_units_5000sqft': info['base_units'],
        'max_stories': info['max_stories'],
        'middle_housing_eligible': info['middle_housing'],
        'hillside_overlay': info['hills'],
        'density_bonus_eligible': info['density_bonus'],
        'description': info['desc'],
    })

df_potential = pd.DataFrame(rows)
df_potential.to_sql('development_potential', conn, if_exists='replace', index=False)

print(f"✅ Saved development_potential table ({len(df_potential)} zone types)\n")

# Summary
print("📊 DEVELOPMENT POTENTIAL SUMMARY:\n")

middle = df_potential[df_potential['middle_housing_eligible']==True]
print(f"   Middle housing eligible zones:  {len(middle)} ({', '.join(middle['zone_class'].tolist())})")

bonus = df_potential[df_potential['density_bonus_eligible']==True]
print(f"   Density bonus eligible zones:   {len(bonus)}")

no_housing = df_potential[df_potential['base_units_5000sqft']==0]
print(f"   No housing allowed zones:       {len(no_housing)} ({', '.join(no_housing['zone_class'].tolist())})")

hills = df_potential[df_potential['hillside_overlay']==True]
print(f"   Hillside overlay zones:         {len(hills)} ({', '.join(hills['zone_class'].tolist())})")

# Show the full table
print(f"\n{'─'*90}")
print(f"{'Zone':<18} {'Base Units':>10} {'Stories':>8} {'MidHousing':>11} {'Hills':>6} {'Bonus':>6}  Description")
print(f"{'─'*90}")
for _, r in df_potential.sort_values('zone_class').iterrows():
    bu = str(r['base_units_5000sqft']) if r['base_units_5000sqft'] is not None else 'varies'
    st = str(r['max_stories']) if r['max_stories'] is not None else 'varies'
    mh = '✅' if r['middle_housing_eligible'] else '—'
    hi = '⛰️' if r['hillside_overlay'] else '—'
    db = '✅' if r['density_bonus_eligible'] else '—'
    print(f"   {r['zone_class']:<15} {bu:>10} {st:>8} {mh:>11} {hi:>6} {db:>6}  {r['description']}")

conn.close()
print(f"\n💾 Saved 'development_potential' table to database")

🏗️ ASSIGNING DEVELOPMENT POTENTIAL TO EVERY PARCEL

✅ Saved development_potential table (41 zone types)

📊 DEVELOPMENT POTENTIAL SUMMARY:

   Middle housing eligible zones:  4 (R-1, R-2, R-2A, MUR)
   Density bonus eligible zones:   35
   No housing allowed zones:       5 (M, MM, MRD, U, X)
   Hillside overlay zones:         10 (R-1H, R-2H, R-2AH, R-3H, R-4H, R-5H, R-SH, C-N(H), C-NS(H), ES-R)

──────────────────────────────────────────────────────────────────────────────────────────
Zone               Base Units  Stories  MidHousing  Hills  Bonus  Description
──────────────────────────────────────────────────────────────────────────────────────────
   C-AC                   nan      5.0           —      —      ✅  Adeline Corridor — mixed use with housing
   C-C                    nan      4.0           —      —      ✅  Corridor Commercial — housing above ground floor
   C-DMU Buffer           nan      5.0           —      —      ✅  Downtown Mixed Use Buffer — housing allowed
   C-DMU 

In [20]:
# CELL: Check if parcels already have zoning info, then assign zones
print("🔗 ASSIGNING ZONING TO PARCELS\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Check what columns parcels already have
cols = pd.read_sql("PRAGMA table_info(parcels_arcgis)", conn)
print("Parcel columns:")
for _, c in cols.iterrows():
    print(f"   {c['name']}")

# Check UseCode distribution — this is from the Assessor, not zoning
print("\nDo parcels have a zoning column already?")
zone_cols = [c['name'] for _, c in cols.iterrows() 
             if 'zone' in c['name'].lower() or 'zoning' in c['name'].lower() or 'district' in c['name'].lower()]
print(f"   Zoning-related columns: {zone_cols or 'NONE'}")

# Check if addresses have zoning
addr_cols = pd.read_sql("PRAGMA table_info(addresses_arcgis)", conn)
addr_zone = [c['name'] for _, c in addr_cols.iterrows() 
             if 'zone' in c['name'].lower() or 'zoning' in c['name'].lower()]
print(f"   Address zoning columns: {addr_zone or 'NONE'}")

# We'll use the ArcGIS Identify endpoint to batch-query zones
# This tells us what zone a given point falls in
# Let's test with one parcel first
sample = pd.read_sql("SELECT APN, LAT_DEG, LONG_DEG FROM parcels_arcgis WHERE LAT_DEG IS NOT NULL LIMIT 1", conn)
print(f"\nTest parcel: APN={sample.iloc[0]['APN']}, lat={sample.iloc[0]['LAT_DEG']}, lon={sample.iloc[0]['LONG_DEG']}")

conn.close()

🔗 ASSIGNING ZONING TO PARCELS

Parcel columns:
   SitusStree
   the_geom
   DATE_UPDAT
   APN
   SitusStr_1
   SitusUnit
   SitusCity
   SitusZip
   UseCode
   BuildingAr
   LotSize
   SitusAddre
   Longitude
   Latitude
   PARCELID
   EXT_MIN_X
   EXT_MIN_Y
   EXT_MAX_X
   EXT_MAX_Y
   corridor
   apn_clean
   apn_norm

Do parcels have a zoning column already?
   Zoning-related columns: NONE
   Address zoning columns: NONE


DatabaseError: Execution failed on sql 'SELECT APN, LAT_DEG, LONG_DEG FROM parcels_arcgis WHERE LAT_DEG IS NOT NULL LIMIT 1': no such column: LAT_DEG

In [21]:
# CELL: Check actual parcel column names and find coordinates
conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

cols = pd.read_sql("PRAGMA table_info(parcels_arcgis)", conn)
print("All parcel columns:")
for _, c in cols.iterrows():
    print(f"   {c['name']}")

# Find coordinate columns
coord_cols = [c['name'] for _, c in cols.iterrows() 
              if any(k in c['name'].lower() for k in ['lat', 'lon', 'long', 'x', 'y', 'coord', 'deg'])]
print(f"\nPossible coordinate columns: {coord_cols}")

# Show a sample row
sample = pd.read_sql("SELECT * FROM parcels_arcgis LIMIT 1", conn)
if coord_cols:
    print(f"\nSample values:")
    for c in coord_cols:
        print(f"   {c}: {sample.iloc[0][c]}")

conn.close()

All parcel columns:
   SitusStree
   the_geom
   DATE_UPDAT
   APN
   SitusStr_1
   SitusUnit
   SitusCity
   SitusZip
   UseCode
   BuildingAr
   LotSize
   SitusAddre
   Longitude
   Latitude
   PARCELID
   EXT_MIN_X
   EXT_MIN_Y
   EXT_MAX_X
   EXT_MAX_Y
   corridor
   apn_clean
   apn_norm

Possible coordinate columns: ['SitusCity', 'Longitude', 'Latitude', 'EXT_MIN_X', 'EXT_MIN_Y', 'EXT_MAX_X', 'EXT_MAX_Y']

Sample values:
   SitusCity: BERKELEY
   Longitude: -122.26636568
   Latitude: 37.85201812
   EXT_MIN_X: 564,525.0572
   EXT_MIN_Y: 4,189,644.3367
   EXT_MAX_X: 564,556.3459
   EXT_MAX_Y: 4,189,655.4693


In [22]:
# CELL: Assign zoning district to each parcel using ArcGIS Identify
print("🗺️ ASSIGNING ZONING DISTRICTS TO PARCELS\n")
print("="*70)

import time

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Load parcels with coordinates
df_p = pd.read_sql("""
    SELECT APN, apn_norm, Latitude, Longitude, SitusAddre, UseCode, LotSize
    FROM parcels_arcgis 
    WHERE Latitude IS NOT NULL AND Longitude IS NOT NULL
      AND Latitude BETWEEN 37.84 AND 37.92
      AND Longitude BETWEEN -122.33 AND -122.23
""", conn)
print(f"Parcels with valid coordinates: {len(df_p):,}")

# ArcGIS Identify endpoint — Layer 24 is Zoning Districts
ARCGIS_BASE = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Accela/MapServer"

# We need to convert lat/lon to the server's coordinate system (UTM 32610)
# But the Identify endpoint accepts geometryType=esriGeometryPoint with 
# inSR=4326 (WGS84 lat/lon)

def get_zone_for_point(lat, lon):
    """Query ArcGIS Identify to find zoning district for a lat/lon point"""
    params = {
        'geometry': f'{lon},{lat}',
        'geometryType': 'esriGeometryPoint',
        'sr': 4326,  # WGS84
        'layers': 'all:24',  # Layer 24 = Zoning Districts
        'tolerance': 2,
        'mapExtent': '-122.33,37.84,-122.23,37.92',
        'imageDisplay': '800,600,96',
        'returnGeometry': 'false',
        'f': 'json'
    }
    try:
        resp = requests.get(f"{ARCGIS_BASE}/identify", params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            results = data.get('results', [])
            if results:
                attrs = results[0].get('attributes', {})
                return attrs.get('ZONECLASS', None)
    except:
        pass
    return None

# Test with one parcel first
test = df_p.iloc[0]
test_zone = get_zone_for_point(test['Latitude'], test['Longitude'])
print(f"\nTest: APN {test['APN']} at ({test['Latitude']:.6f}, {test['Longitude']:.6f})")
print(f"  Address: {test['SitusAddre']}")
print(f"  Zone: {test_zone}")

if test_zone:
    print(f"\n✅ Identify endpoint works! Now processing all {len(df_p):,} parcels...")
    print("   (This will take ~15-20 minutes at ~30 parcels/sec)\n")
    
    zones = []
    errors = 0
    start_time = time.time()
    
    for i, row in df_p.iterrows():
        zone = get_zone_for_point(row['Latitude'], row['Longitude'])
        zones.append(zone)
        
        if zone is None:
            errors += 1
        
        # Progress every 500 parcels
        if (len(zones)) % 500 == 0:
            elapsed = time.time() - start_time
            rate = len(zones) / elapsed
            remaining = (len(df_p) - len(zones)) / rate / 60
            pct = len(zones) / len(df_p) * 100
            found = sum(1 for z in zones if z)
            print(f"   {len(zones):,}/{len(df_p):,} ({pct:.0f}%) — {found:,} zoned, {errors} errors — ~{remaining:.0f} min remaining")
        
        # Small delay to be polite
        if len(zones) % 10 == 0:
            time.sleep(0.1)
    
    df_p['zone_class'] = zones
    
    elapsed = time.time() - start_time
    found = df_p['zone_class'].notna().sum()
    print(f"\n{'='*70}")
    print(f"✅ DONE in {elapsed/60:.1f} minutes")
    print(f"   Zoned: {found:,}/{len(df_p):,} ({found/len(df_p)*100:.1f}%)")
    print(f"   Errors: {errors:,}")
    
    # Save to database
    df_p[['apn_norm', 'zone_class']].to_sql('parcel_zones', conn, if_exists='replace', index=False)
    conn.execute('CREATE INDEX IF NOT EXISTS idx_pz_apn ON parcel_zones(apn_norm)')
    conn.commit()
    
    # Show distribution
    print(f"\n📋 Zone distribution:")
    for zone, cnt in df_p['zone_class'].value_counts().head(15).items():
        print(f"   {zone}: {cnt:,} parcels")
    
    print(f"\n💾 Saved 'parcel_zones' table")
else:
    print("❌ Identify endpoint failed. Check the ArcGIS server.")

conn.close()

🗺️ ASSIGNING ZONING DISTRICTS TO PARCELS

Parcels with valid coordinates: 0


IndexError: single positional indexer is out-of-bounds

In [23]:
# CELL: Debug — check coordinate values in parcels_arcgis
conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# How many parcels have coordinates?
stats = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN Latitude IS NOT NULL THEN 1 ELSE 0 END) as has_lat,
        SUM(CASE WHEN Longitude IS NOT NULL THEN 1 ELSE 0 END) as has_lon,
        MIN(Latitude) as min_lat, MAX(Latitude) as max_lat,
        MIN(Longitude) as min_lon, MAX(Longitude) as max_lon
    FROM parcels_arcgis
""", conn)

print("Parcels coordinate stats:")
for col in stats.columns:
    print(f"   {col}: {stats.iloc[0][col]}")

# Sample a few
sample = pd.read_sql("""
    SELECT APN, Latitude, Longitude, SitusAddre 
    FROM parcels_arcgis 
    WHERE Latitude IS NOT NULL 
    LIMIT 5
""", conn)
print(f"\nSample parcels:")
display(sample)

conn.close()

Parcels coordinate stats:
   total: 29024
   has_lat: 29024
   has_lon: 29024
   min_lat: 37.84599202
   max_lat: 37.90536047
   min_lon: -122.23448134
   max_lon: -122.33722447

Sample parcels:


,APN,Latitude,Longitude,SitusAddre
0,16-1428-2-2,37.85201812,-122.26636568,3208 SHATTUCK AVE BERKELEY 94705
1,16-1425-57,37.85214246,-122.2653821,6618 SHATTUCK AVE BERKELEY 94609
2,16-1422-22,37.85275106,-122.26088766,2320 WOOLSEY ST BERKELEY 94705
3,16-1422-20,37.8527926,-122.26057139,6699 TELEGRAPH AVE BERKELEY 94609
4,16-1422-24,37.85273147,-122.26103804,2314 WOOLSEY ST BERKELEY 94705


In [24]:
# CELL: Test ArcGIS Identify with one parcel, then batch all
print("🗺️ TESTING ZONING LOOKUP\n")
print("="*70)

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

ARCGIS_BASE = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Accela/MapServer"

def get_zone_for_point(lat, lon):
    params = {
        'geometry': f'{lon},{lat}',
        'geometryType': 'esriGeometryPoint',
        'sr': 4326,
        'layers': 'all:24',
        'tolerance': 2,
        'mapExtent': '-122.34,37.84,-122.23,37.91',
        'imageDisplay': '800,600,96',
        'returnGeometry': 'false',
        'f': 'json'
    }
    try:
        resp = requests.get(f"{ARCGIS_BASE}/identify", params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            results = data.get('results', [])
            if results:
                return results[0].get('attributes', {}).get('ZONECLASS', None)
    except:
        pass
    return None

# Test with first 5 parcels
sample = pd.read_sql("SELECT APN, Latitude, Longitude, SitusAddre FROM parcels_arcgis LIMIT 5", conn)

print("Testing 5 parcels:\n")
for _, row in sample.iterrows():
    zone = get_zone_for_point(row['Latitude'], row['Longitude'])
    print(f"   {row['APN']:>15}  {row['SitusAddre']:<40}  → {zone or '???'}")

conn.close()

🗺️ TESTING ZONING LOOKUP

Testing 5 parcels:

       16-1428-2-2  3208 SHATTUCK AVE BERKELEY 94705          → C-SA
        16-1425-57  6618 SHATTUCK AVE BERKELEY 94609          → C-SA
        16-1422-22  2320 WOOLSEY ST BERKELEY 94705            → C-C
        16-1422-20  6699 TELEGRAPH AVE BERKELEY 94609         → C-C
        16-1422-24  2314 WOOLSEY ST BERKELEY 94705            → C-C


In [25]:
# CELL: Batch assign zoning to ALL 29,024 parcels
print("🗺️ ASSIGNING ZONING TO ALL PARCELS\n")
print("="*70)
print("⏱️ Estimated time: 15-25 minutes\n")

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

df_p = pd.read_sql("""
    SELECT APN, apn_norm, Latitude, Longitude, SitusAddre, UseCode, LotSize
    FROM parcels_arcgis 
    WHERE Latitude IS NOT NULL AND Longitude IS NOT NULL
""", conn)
print(f"Parcels to process: {len(df_p):,}")

zones = []
errors = 0
start_time = time.time()

for i, row in df_p.iterrows():
    zone = get_zone_for_point(row['Latitude'], row['Longitude'])
    zones.append(zone)
    
    if zone is None:
        errors += 1
    
    count = len(zones)
    if count % 500 == 0:
        elapsed = time.time() - start_time
        rate = count / elapsed
        remaining = (len(df_p) - count) / rate / 60
        found = sum(1 for z in zones if z)
        print(f"   {count:,}/{len(df_p):,} ({count/len(df_p)*100:.0f}%) — "
              f"{found:,} zoned, {errors} errors — "
              f"{rate:.0f}/sec — ~{remaining:.0f} min left")
    
    # Throttle slightly
    if count % 10 == 0:
        time.sleep(0.05)

df_p['zone_class'] = zones

elapsed = time.time() - start_time
found = df_p['zone_class'].notna().sum()
print(f"\n{'='*70}")
print(f"✅ DONE in {elapsed/60:.1f} minutes")
print(f"   Zoned:  {found:,}/{len(df_p):,} ({found/len(df_p)*100:.1f}%)")
print(f"   Errors: {errors:,}")

# Save parcel-zone mapping
df_p[['apn_norm', 'zone_class']].to_sql('parcel_zones', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX IF NOT EXISTS idx_pz_apn ON parcel_zones(apn_norm)')

# Also create the master view joining everything
conn.execute("DROP VIEW IF EXISTS parcels_full")
conn.execute("""
    CREATE VIEW parcels_full AS
    SELECT 
        p.*,
        pz.zone_class,
        dp.base_units_5000sqft,
        dp.max_stories,
        dp.middle_housing_eligible,
        dp.hillside_overlay,
        dp.density_bonus_eligible,
        dp.description as zone_description,
        a.FullAddress,
        a.OwnerName,
        a.OwnerAddress1,
        a.ownercityst,
        a.BldgSqft as addr_BldgSqft
    FROM parcels_arcgis p
    LEFT JOIN parcel_zones pz ON p.apn_norm = pz.apn_norm
    LEFT JOIN addresses_arcgis a ON p.apn_norm = a.apn_norm
    LEFT JOIN development_potential dp ON pz.zone_class = dp.zone_class
""")
conn.commit()

# Summary
print(f"\n📋 Zone distribution:")
for zone, cnt in df_p['zone_class'].value_counts().head(20).items():
    print(f"   {zone:>15}: {cnt:,} parcels")

print(f"\n👁️ Created master view 'parcels_full' joining parcels + zones + addresses + development potential")

conn.close()
print(f"\n💾 All saved to database")

🗺️ ASSIGNING ZONING TO ALL PARCELS

⏱️ Estimated time: 15-25 minutes

Parcels to process: 29,024
   500/29,024 (2%) — 500 zoned, 0 errors — 2/sec — ~301 min left
   1,000/29,024 (3%) — 1,000 zoned, 0 errors — 2/sec — ~295 min left
   1,500/29,024 (5%) — 1,499 zoned, 1 errors — 1/sec — ~463 min left
   2,000/29,024 (7%) — 1,999 zoned, 1 errors — 1/sec — ~410 min left
   2,500/29,024 (9%) — 2,497 zoned, 3 errors — 1/sec — ~376 min left
   3,000/29,024 (10%) — 2,997 zoned, 3 errors — 1/sec — ~351 min left
   3,500/29,024 (12%) — 3,497 zoned, 3 errors — 1/sec — ~333 min left
   4,000/29,024 (14%) — 3,996 zoned, 4 errors — 1/sec — ~317 min left
   4,500/29,024 (16%) — 4,496 zoned, 4 errors — 1/sec — ~305 min left
   5,000/29,024 (17%) — 4,996 zoned, 4 errors — 1/sec — ~293 min left
   5,500/29,024 (19%) — 5,496 zoned, 4 errors — 1/sec — ~283 min left
   6,000/29,024 (21%) — 5,996 zoned, 4 errors — 1/sec — ~273 min left
   6,500/29,024 (22%) — 6,496 zoned, 4 errors — 1/sec — ~265 min left
  

In [ ]:
# CELL 6: Fetch permits from discovered datasets
# Update the PERMIT_DATASET_ID below based on what Cell 5 found.
# Common Berkeley permit datasets:
#   - Building permits
#   - Zoning permits
#   - Planning applications

print("🏗️ FETCHING PERMIT RECORDS\n")
print("="*70)

# Try fetching from each permit dataset found above
all_permits = []

for ds in permit_datasets:
    dataset_id = ds['id']
    name = ds['name']
    
    try:
        # First get a small sample to see the columns
        url = f"{BASE_URL}/{dataset_id}.json?$limit=5"
        if APP_TOKEN:
            url += f"&$$app_token={APP_TOKEN}"
        
        response = requests.get(url, timeout=15)
        
        if response.status_code == 200:
            sample = response.json()
            if sample:
                cols = list(sample[0].keys())
                
                # Check if this dataset has address or APN fields
                has_address = any('addr' in c.lower() or 'location' in c.lower() or 'street' in c.lower() for c in cols)
                has_apn = any('apn' in c.lower() or 'parcel' in c.lower() for c in cols)
                has_permit = any('permit' in c.lower() or 'status' in c.lower() or 'type' in c.lower() for c in cols)
                
                print(f"\n📁 {name} ({dataset_id})")
                print(f"   Columns: {cols[:10]}{'...' if len(cols) > 10 else ''}")
                print(f"   Has address: {has_address} | Has APN: {has_apn} | Has permit info: {has_permit}")
                
                if has_permit and (has_address or has_apn):
                    print(f"   ✅ RELEVANT — fetching all records...")
                    
                    # Fetch all records from this dataset
                    records = []
                    offset = 0
                    limit = 1000
                    
                    while True:
                        fetch_url = f"{BASE_URL}/{dataset_id}.json?$limit={limit}&$offset={offset}"
                        if APP_TOKEN:
                            fetch_url += f"&$$app_token={APP_TOKEN}"
                        
                        resp = requests.get(fetch_url, timeout=30)
                        if resp.status_code != 200:
                            break
                        
                        batch = resp.json()
                        if not batch:
                            break
                        
                        records.extend(batch)
                        offset += limit
                        
                        if len(batch) < limit:
                            break
                    
                    df_temp = pd.DataFrame(records)
                    df_temp['_source_dataset'] = name
                    df_temp['_source_id'] = dataset_id
                    all_permits.append(df_temp)
                    print(f"   📊 Fetched {len(df_temp):,} records")
                else:
                    print(f"   ⏭️ Skipping (not relevant)")
            else:
                print(f"\n📁 {name} — empty dataset")
        else:
            print(f"\n📁 {name} — HTTP {response.status_code}")
    
    except Exception as e:
        print(f"\n📁 {name} — Error: {e}")

print(f"\n{'='*70}")
print(f"\n📊 SUMMARY: Found {len(all_permits)} relevant permit datasets")
for df in all_permits:
    src = df['_source_dataset'].iloc[0]
    print(f"   • {src}: {len(df):,} records")

In [ ]:
# CELL 7: Also load housing pipeline projects from local CSV
print("🏠 LOADING HOUSING PIPELINE PROJECTS\n")
print("="*70)

import glob

# Find the most recent housing projects file
patterns = [
    '/Users/johngage/berkeley-data/*housing*project*.csv',
    '/Users/johngage/berkeley-data/*pipeline*.csv',
    '/Users/johngage/berkeley-data/*permits*.csv',
]

found_files = []
for pattern in patterns:
    found_files.extend(glob.glob(pattern))

if found_files:
    print(f"Found {len(found_files)} local data files:\n")
    for f in sorted(found_files, key=os.path.getmtime, reverse=True):
        size = os.path.getsize(f) / 1024
        mtime = datetime.fromtimestamp(os.path.getmtime(f)).strftime('%Y-%m-%d')
        print(f"  📄 {os.path.basename(f)} ({size:.0f} KB, modified {mtime})")
    
    # Load the most recent one
    latest = max(found_files, key=os.path.getmtime)
    df_pipeline = pd.read_csv(latest)
    print(f"\n✅ Loaded: {os.path.basename(latest)}")
    print(f"   {len(df_pipeline)} projects")
    print(f"   Columns: {df_pipeline.columns.tolist()}")
else:
    print("⚠️ No local housing project CSVs found.")
    print("   We'll rely on API permit data from Cell 6.")
    df_pipeline = None

# Also check the SQLite database for housing tables
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
housing_tables = [t for t in tables['name'].tolist() if any(kw in t.lower() for kw in ['housing', 'permit', 'pipeline', 'project'])]
if housing_tables:
    print(f"\n📊 Housing-related tables in SQLite: {housing_tables}")
    for t in housing_tables:
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM [{t}]", conn).iloc[0]['n']
        print(f"   • {t}: {count:,} rows")
conn.close()

In [ ]:
# CELL 8: Join parcels with permits to find active housing parcels
print("🔗 JOINING PARCELS WITH ACTIVE PERMITS\n")
print("="*70)

# We'll try to join on APN first, then fall back to address matching

# Combine all permit dataframes
if all_permits:
    df_all_permits = pd.concat(all_permits, ignore_index=True)
    print(f"Total permit records: {len(df_all_permits):,}")
    
    # Find APN column in permits
    permit_apn_cols = [c for c in df_all_permits.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
    permit_addr_cols = [c for c in df_all_permits.columns if 'addr' in c.lower() or 'location' in c.lower() or 'street' in c.lower()]
    permit_status_cols = [c for c in df_all_permits.columns if 'status' in c.lower()]
    permit_type_cols = [c for c in df_all_permits.columns if 'type' in c.lower() or 'category' in c.lower() or 'description' in c.lower()]
    
    print(f"\nPermit APN columns: {permit_apn_cols}")
    print(f"Permit address columns: {permit_addr_cols}")
    print(f"Permit status columns: {permit_status_cols}")
    print(f"Permit type columns: {permit_type_cols}")
    
    # Filter for housing-related permits
    # Look for keywords in type/description columns
    housing_keywords = ['residential', 'housing', 'dwelling', 'apartment', 'condo',
                        'ADU', 'accessory', 'multi-family', 'new construction',
                        'addition', 'unit', 'SB 9', 'duplex', 'triplex']
    
    # Check each type/description column for housing keywords
    housing_mask = pd.Series([False] * len(df_all_permits))
    
    for col in permit_type_cols:
        col_text = df_all_permits[col].astype(str).str.lower()
        for kw in housing_keywords:
            housing_mask = housing_mask | col_text.str.contains(kw.lower(), na=False)
    
    df_housing_permits = df_all_permits[housing_mask].copy()
    print(f"\n🏠 Housing-related permits: {len(df_housing_permits):,} out of {len(df_all_permits):,}")
    
    # Filter for active status
    if permit_status_cols:
        status_col = permit_status_cols[0]
        print(f"\n📊 Status distribution ('{status_col}'):")
        print(df_housing_permits[status_col].value_counts().head(15).to_string())
        
        # Active statuses (adjust based on what you see above)
        active_keywords = ['open', 'active', 'issued', 'approved', 'pending',
                          'under review', 'in progress', 'submitted', 'applied']
        
        active_mask = df_housing_permits[status_col].astype(str).str.lower().apply(
            lambda x: any(kw in x for kw in active_keywords)
        )
        
        df_active = df_housing_permits[active_mask].copy()
        print(f"\n✅ ACTIVE housing permits: {len(df_active):,}")
    else:
        df_active = df_housing_permits
        print("\n⚠️ No status column found — using all housing permits")
else:
    print("⚠️ No permit data from API. Using local pipeline data if available.")
    df_active = df_pipeline if df_pipeline is not None else pd.DataFrame()

In [ ]:
# CELL 9: Match active permits to parcels
print("🗺️ MATCHING ACTIVE PERMITS TO PARCELS\n")
print("="*70)

if len(df_active) > 0 and len(df_parcels) > 0:
    
    # Try APN join first
    permit_apn_cols = [c for c in df_active.columns if 'apn' in c.lower() or 'parcel' in c.lower()]
    
    if permit_apn_cols:
        PERMIT_APN = permit_apn_cols[0]
        df_active['apn_clean'] = df_active[PERMIT_APN].astype(str).str.strip().str.upper()
        
        # Join
        df_matched = df_parcels.merge(
            df_active,
            on='apn_clean',
            how='inner',
            suffixes=('_parcel', '_permit')
        )
        
        print(f"✅ Matched {len(df_matched):,} permit-parcel combinations")
        print(f"   Unique parcels with active housing permits: {df_matched['apn_clean'].nunique():,}")
        print(f"   Out of {len(df_parcels):,} total parcels ({df_matched['apn_clean'].nunique()/len(df_parcels)*100:.2f}%)")
    else:
        print("⚠️ No APN column in permits — trying address matching...")
        # Address matching fallback would go here
        df_matched = pd.DataFrame()
    
    # Summary
    if len(df_matched) > 0:
        print(f"\n{'='*70}")
        print(f"\n📊 SUMMARY OF PARCELS WITH ACTIVE HOUSING PERMITS:\n")
        
        # Show sample
        display_cols = [c for c in df_matched.columns if any(kw in c.lower() for kw in 
                        ['apn', 'addr', 'situs', 'status', 'type', 'description', 'unit', 'date'])]
        if display_cols:
            print(df_matched[display_cols[:8]].head(20).to_string())
        else:
            display(df_matched.head(20))
        
        # Save results
        output_path = '/Users/johngage/berkeley-data/parcels_with_active_housing_permits.csv'
        df_matched.to_csv(output_path, index=False)
        print(f"\n💾 Saved to: {output_path}")
        
        # Also save to SQLite
        conn = sqlite3.connect(DB_PATH)
        df_matched.to_sql('parcels_active_housing', conn, if_exists='replace', index=False)
        conn.close()
        print(f"💾 Saved to SQLite table: parcels_active_housing")
else:
    print("⚠️ No active permits or parcels to match.")

In [ ]:
# CELL 10: Quick map of parcels with active housing permits
print("🗺️ MAPPING PARCELS WITH ACTIVE HOUSING PERMITS\n")

try:
    import folium
    from folium.plugins import MarkerCluster
    
    # Berkeley center
    m = folium.Map(location=[37.8716, -122.2727], zoom_start=14, 
                   tiles='CartoDB positron')
    
    # Find lat/lon columns
    lat_cols = [c for c in df_matched.columns if 'lat' in c.lower()]
    lon_cols = [c for c in df_matched.columns if 'lon' in c.lower() or 'lng' in c.lower()]
    
    if lat_cols and lon_cols:
        LAT = lat_cols[0]
        LON = lon_cols[0]
        
        # Filter rows with valid coordinates
        df_map = df_matched.dropna(subset=[LAT, LON]).copy()
        df_map[LAT] = pd.to_numeric(df_map[LAT], errors='coerce')
        df_map[LON] = pd.to_numeric(df_map[LON], errors='coerce')
        df_map = df_map.dropna(subset=[LAT, LON])
        
        print(f"Mapping {len(df_map):,} parcels with coordinates...")
        
        cluster = MarkerCluster()
        
        for _, row in df_map.iterrows():
            addr = row.get(ADDR_COL, row.get('apn_clean', 'Unknown'))
            popup_text = f"<b>{addr}</b><br>APN: {row.get('apn_clean', 'N/A')}"
            
            folium.CircleMarker(
                location=[row[LAT], row[LON]],
                radius=6,
                color='#2d5f3a',
                fill=True,
                fill_color='#4CAF50',
                fill_opacity=0.7,
                popup=folium.Popup(popup_text, max_width=250)
            ).add_to(cluster)
        
        cluster.add_to(m)
        
        # Save map
        map_path = '/Users/johngage/berkeley-data/active_housing_permits_map.html'
        m.save(map_path)
        print(f"\n💾 Map saved to: {map_path}")
        
        display(m)
    else:
        print("⚠️ No coordinate columns found in matched data.")
        print(f"   Available columns: {df_matched.columns.tolist()}")
        print("   You may need to geocode the addresses first.")
        
except ImportError:
    print("⚠️ folium not installed. Run: pip install folium")
    print("   Skipping map — data is still saved to CSV and SQLite.")

In [ ]:
# CELL 11: Final summary
print("\n" + "="*70)
print("📊 FINAL SUMMARY")
print("="*70)
print(f"\n  Total Berkeley parcels:          {len(df_parcels):>8,}")

if 'df_all_permits' in dir() and len(df_all_permits) > 0:
    print(f"  Total permit records fetched:     {len(df_all_permits):>8,}")

if 'df_housing_permits' in dir() and len(df_housing_permits) > 0:
    print(f"  Housing-related permits:          {len(df_housing_permits):>8,}")

if 'df_active' in dir() and len(df_active) > 0:
    print(f"  Active housing permits:           {len(df_active):>8,}")

if 'df_matched' in dir() and len(df_matched) > 0:
    print(f"  Parcels with active permits:      {df_matched['apn_clean'].nunique():>8,}")
    pct = df_matched['apn_clean'].nunique() / len(df_parcels) * 100
    print(f"  Percentage of all parcels:        {pct:>7.2f}%")

print(f"\n  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n{'='*70}")
print("\n✅ Done! Results saved to:")
print(f"   CSV: /Users/johngage/berkeley-data/parcels_with_active_housing_permits.csv")
print(f"   SQLite: {DB_PATH} → table 'parcels_active_housing'")
print(f"   Map: /Users/johngage/berkeley-data/active_housing_permits_map.html")